# Tahap 2: Ekstraksi dan Pembersihan Dokumen Pedoman Gizi
Notebook ini mengekstrak materi penting dari dokumen PDF referensi gizi (Skripsi UII Bab 2.1 dan Lampiran D, Permenkes No. 28/2019, dan Permenkes No. 41/2014) serta membersihkan header dan footer berulang.

In [1]:
import os
import re
import sys
import pypdf

if sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

def clean_page_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if re.match(r'^-?\s*\d+\s*-?$', stripped):
            continue
        if re.search(r'(BERITA NEGARA REPUBLIK INDONESIA|UNIVERSITAS ISLAM INDONESIA|www\.peraturan\.go\.id|JDIH KEMENKES)', stripped, re.IGNORECASE):
            continue
        cleaned_lines.append(line)
    cleaned_text = '\n'.join(cleaned_lines)
    return re.sub(r'\n{3,}', '\n\n', cleaned_text).strip()

In [2]:
skripsi_pdf = "[1] SISTEM ANALISIS POLA KONSUMSI DAN REKOMENDASI DIET BERBASIS GEN AI .pdf"
if not os.path.exists(skripsi_pdf):
    skripsi_pdf = os.path.join("doc", skripsi_pdf)

reader = pypdf.PdfReader(skripsi_pdf)

# Bab 2.1 halaman 20 sampai 25 (index 19-24)
bab2_pages = [clean_page_text(reader.pages[p].extract_text() or '') for p in range(19, 25)]
bab2_text = "\n\n".join(bab2_pages)

# Lampiran D halaman 94 sampai 96 (index 93-95)
lampiran_d_pages = [clean_page_text(reader.pages[p].extract_text() or '') for p in range(93, 96)]
lampiran_d_text = "\n\n".join(lampiran_d_pages)

skripsi_md = f"""# Ringkasan Landasan Teori Gizi & Rumus Perhitungan Diet (Skripsi UII)
**Sumber:** *Sistem Analisis Pola Konsumsi dan Rekomendasi Diet Berbasis Generative AI*

---

## BAB 2.1 Konsep Dasar Gizi dan Penilaian Konsumsi

{bab2_text}

---

## LAMPIRAN D: Implementasi Rumus Perhitungan Kebutuhan Nutrisi

```typescript
{lampiran_d_text}
```

---

### Ringkasan Parameter & Rumus Utama:
1. **Indeks Massa Tubuh (IMT / BMI):**
   - Rumus: $\\text{{IMT}} = \\frac{{\\text{{Berat Badan (kg)}}}}{{(\\text{{Tinggi Badan (m)}})^2}}$
   - Klasifikasi Kemenkes:
     * Sangat Kurus (Kekurangan BB tingkat berat): $< 17.0$
     * Kurus (Kekurangan BB tingkat ringan): $17.0 - 18.4$
     * Normal: $18.5 - 25.0$
     * Gemuk (Kelebihan BB tingkat ringan): $25.1 - 27.0$
     * Obesitas (Kelebihan BB tingkat berat): $> 27.0$

2. **Basal Metabolic Rate (BMR) - Persamaan Mifflin-St Jeor:**
   - **Pria:** $\\text{{BMR}} = (10 \\times \\text{{BB}}) + (6.25 \\times \\text{{TB}}) - (5 \\times U) + 5$
   - **Wanita:** $\\text{{BMR}} = (10 \\times \\text{{BB}}) + (6.25 \\times \\text{{TB}}) - (5 \\times U) - 161$
   *(BB dalam kg, TB dalam cm, U dalam tahun)*

3. **Total Daily Energy Expenditure (TDEE):**
   - Rumus: $\\text{{TDEE}} = \\text{{BMR}} \\times \\text{{PAL}}$
   - **Physical Activity Level (PAL):**
     * Sedentari (minim aktivitas): 1.2
     * Aktivitas Ringan (olahraga 1-3 hari/minggu): 1.375
     * Aktivitas Sedang (olahraga 3-5 hari/minggu): 1.55
     * Sangat Aktif (olahraga berat 6-7 hari/minggu): 1.725
     * Ekstra Aktif (pekerja fisik/atlet): 1.9

4. **Distribusi Kebutuhan Makronutrien:**
   - **Karbohidrat:** 45% - 65% dari total kalori (1 gram karbohidrat = 4 kkal)
   - **Protein:** 10% - 20% (atau 1.2 - 2.0 g/kg BB sesuai target) (1 gram protein = 4 kkal)
   - **Lemak:** 20% - 30% dari total kalori (1 gram lemak = 9 kkal)
"""

os.makedirs("doc", exist_ok=True)
skripsi_out = os.path.join("doc", "extracted_bab2_pedoman_gizi.md")
with open(skripsi_out, "w", encoding="utf-8") as f:
    f.write(skripsi_md)
print(f"Skripsi Bab 2.1 dan Lampiran D disimpan ke {skripsi_out} ({len(skripsi_md)} karakter)")

Skripsi Bab 2.1 dan Lampiran D disimpan ke doc\extracted_bab2_pedoman_gizi.md (20478 karakter)


In [3]:
# Permenkes 28/2019
p2019_pdf = "2019permenkes028.pdf"
if not os.path.exists(p2019_pdf):
    p2019_pdf = os.path.join("doc", p2019_pdf)
r_2019 = pypdf.PdfReader(p2019_pdf)

p2019_articles = "\n\n".join([clean_page_text(r_2019.pages[p].extract_text() or '') for p in range(1, 5)])
p2019_tables = "\n\n".join([clean_page_text(r_2019.pages[p].extract_text() or '') for p in range(5, 12)])
p2019_usage = "\n\n".join([clean_page_text(r_2019.pages[p].extract_text() or '') for p in range(14, 20)])

p2019_md = f"""# Pedoman Angka Kecukupan Gizi (AKG) 2019
**Sumber:** *Peraturan Menteri Kesehatan Republik Indonesia Nomor 28 Tahun 2019 tentang Angka Kecukupan Gizi yang Dianjurkan untuk Masyarakat Indonesia*

---

## 1. Definisi dan Ketentuan Umum
{p2019_articles}

---

## 2. Tabel Angka Kecukupan Gizi Rata-rata yang Dianjurkan (Per Orang Per Hari)
{p2019_tables}

---

## 3. Petunjuk Penggunaan AKG Individu dan Kelompok
{p2019_usage}
"""
p2019_out = os.path.join("doc", "extracted_permenkes_akg_2019.md")
with open(p2019_out, "w", encoding="utf-8") as f:
    f.write(p2019_md)
print(f"Permenkes 28/2019 disimpan ke {p2019_out} ({len(p2019_md)} karakter)")

# Permenkes 41/2014
p2014_pdf = "2014permenkes041.pdf"
if not os.path.exists(p2014_pdf):
    p2014_pdf = os.path.join("doc", p2014_pdf)
r_2014 = pypdf.PdfReader(p2014_pdf)

p2014_pilar = "\n\n".join([clean_page_text(r_2014.pages[p].extract_text() or '') for p in range(16, 25)])
p2014_pesan = "\n\n".join([clean_page_text(r_2014.pages[p].extract_text() or '') for p in range(30, 39)])
p2014_visual = "\n\n".join([clean_page_text(r_2014.pages[p].extract_text() or '') for p in range(80, 86)])

p2014_md = f"""# Pedoman Gizi Seimbang (PGS) 2014
**Sumber:** *Peraturan Menteri Kesehatan Republik Indonesia Nomor 41 Tahun 2014 tentang Pedoman Gizi Seimbang*

---

## 1. Empat Pilar Gizi Seimbang
Prinsip Gizi Seimbang terdiri dari 4 (empat) Pilar yang pada dasarnya merupakan rangkaian upaya untuk menyeimbangkan antara zat gizi yang keluar dan zat gizi yang masuk dengan memantau berat badan secara teratur:
1. Membiasakan mengonsumsi aneka ragam makanan.
2. Membiasakan perilaku hidup bersih.
3. Melakukan aktivitas fisik secara teratur.
4. Memantau dan mempertahankan berat badan dalam batas normal.

{p2014_pilar}

---

## 2. Sepuluh Pesan Umum Pedoman Gizi Seimbang
1. Syukuri dan nikmati aneka ragam makanan.
2. Banyak makan sayuran dan cukup buah-buahan.
3. Biasakan mengonsumsi lauk pauk yang mengandung protein tinggi.
4. Biasakan mengonsumsi aneka ragam makanan pokok.
5. Batasi konsumsi pangan manis, asin dan berlemak.
6. Biasakan Sarapan.
7. Biasakan minum air putih yang cukup dan aman.
8. Biasakan membaca label pada kemasan pangan.
9. Cuci tangan pakai sabun dengan air bersih mengalir.
10. Lakukan aktivitas fisik yang cukup dan pertahankan berat badan normal.

{p2014_pesan}

---

## 3. Visualisasi Panduan Piring Makanku & Tumpeng Gizi Seimbang
Panduan porsi sekali makan (Piring Makanku):
- 1/2 bagian piring: terdiri dari 2/3 sayuran dan 1/3 buah-buahan.
- 1/2 bagian piring lainnya: terdiri dari 2/3 makanan pokok (karbohidrat) dan 1/3 lauk-pauk (protein hewani dan nabati).
- Disertai dengan mencuci tangan sebelum makan, minum air putih minimal 8 gelas per hari, dan aktivitas fisik 30 menit per hari.

{p2014_visual}
"""
p2014_out = os.path.join("doc", "extracted_permenkes_gizi_seimbang_2014.md")
with open(p2014_out, "w", encoding="utf-8") as f:
    f.write(p2014_md)
print(f"Permenkes 41/2014 disimpan ke {p2014_out} ({len(p2014_md)} karakter)")

Permenkes 28/2019 disimpan ke doc\extracted_permenkes_akg_2019.md (21920 karakter)


Permenkes 41/2014 disimpan ke doc\extracted_permenkes_gizi_seimbang_2014.md (32881 karakter)


In [4]:
combined_md = f"{skripsi_md}\n\n{'='*40}\n\n{p2014_md}\n\n{'='*40}\n\n{p2019_md}"
combined_out = os.path.join("doc", "pedoman_gizi_lengkap.md")
with open(combined_out, "w", encoding="utf-8") as f:
    f.write(combined_md)

print(f"Dokumen gabungan tersimpan di {combined_out}")
for fname in os.listdir("doc"):
    p = os.path.join("doc", fname)
    print(f"- {fname}: {os.path.getsize(p):,} bytes")

Dokumen gabungan tersimpan di doc\pedoman_gizi_lengkap.md
- extracted_bab2_pedoman_gizi.md: 21,123 bytes
- extracted_permenkes_akg_2019.md: 22,693 bytes
- extracted_permenkes_gizi_seimbang_2014.md: 33,690 bytes
- pedoman_gizi_lengkap.md: 77,602 bytes
